In [3]:
# generate landmark mask
import os
import numpy as np
from PIL import Image
import imageio
import math
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt

def distance_transform(mask):
    """
    Compute the distance transform of a binary mask.

    Parameters:
    - mask: numpy array, binary mask

    Returns:
    - dist_transform: numpy array, distance transform of the mask
    """
    dist_transform = distance_transform_edt(mask)
    return dist_transform


if __name__ == "__main__":

    coords_folder = '/playpen-raid2/qinliu/data/ecDNA/coords'
    masks_folder = '/playpen-raid2/qinliu/data/ecDNA/masks_v2'
    images_folder = '/playpen-raid2/qinliu/data/ecDNA/images'
    files = sorted(os.listdir(images_folder))
    r1, r2, r3 = 3, 5, 100 # inclusive

    # for file in files:
    for i in tqdm(range(len(files))):
        file = files[i]
        file_path = os.path.join(images_folder, file)

        # generate csv files
        raw_name = file.split('.')[0]
        image_npy = np.array(Image.open(file_path))

        mask_npy = np.zeros_like(image_npy) + 1
        H, W = mask_npy.shape
        coords = np.load(f'{coords_folder}//{raw_name}.npy')
        for coord in coords:
            x, y = int(round(coord[0])), int(round(coord[1]))
            for i in range(-r2, r2 + 1):
                for j in range(-r2, r2 + 1):
                    idx, idy = x + i, y + j                
                    if idx >= 0 and idx < W and idy >= 0 and idy < H:
                        dist = math.sqrt((idx - coord[0])**2 + (idy - coord[1])**2)
                        if dist <= r1:
                            mask_npy[idy, idx] = 0

        dist = distance_transform(mask_npy)
        mask_npy = mask_npy * 0 + 128
        mask_npy[dist <= r3] = 0
        mask_npy[dist <= (r2 - r1)] = 192
        mask_npy[dist == 0] = 255
        imageio.imwrite(f'{masks_folder}//{raw_name}.png', mask_npy)

  0%|          | 0/2990 [00:00<?, ?it/s]

100%|██████████| 2990/2990 [26:33<00:00,  1.88it/s]
